# Affinage v2 — corrections et sélection de modèle

Version précédente : **48,0** au niveau abstract, contre 48,5 pour VeriSci.
Deux défauts identifiés par les contrôles par module, corrigés ici.

## Ce qui était faux

**Le classifieur d'étiquette était entraîné sur des entrées irréalistes.** Pour
les abstracts sans preuve, il recevait *les trois premières phrases*. Il a donc
appris « trois premières phrases = NOINFO », un raccourci qui donnait 94,4 %
d'exactitude en isolé — contre 75,7 publié — et qui ne survivait pas à la chaîne
complète, où les phrases viennent du sélecteur. **Le contrôle mesurait un
artefact.** Ici, le classifieur est entraîné sur les sorties réelles du
sélecteur, donc dans les conditions exactes de l'inférence.

**Le seuil était réglé sur le mauvais objectif** : le F1 de sélection de phrases,
alors que la métrique qui compte est le F1 au niveau abstract. D'où un profil
inversé par rapport à VeriSci — P 41,2 / R 57,4 chez nous, P 52,6 / R 45,1 chez
eux. Ici le balayage optimise la métrique finale, sur `val`.

**Le sélecteur était sous-dimensionné** : F1 65,0 contre 72,1 pour RoBERTa-large
et 74,4 pour SciBERT. C'est le vrai goulot. Ici, deux candidats sont entraînés et
le meilleur est retenu **sur `val`**.

## Limites connues, dites maintenant

Pour qu'on n'y revienne pas après coup :

- **`dev` sert au rapport, jamais au réglage.** Tout se règle sur 15 % de `train`
  tenus à l'écart. Le papier, lui, règle son seuil sur `dev` (leur annexe A.1) :
  la comparaison nous est donc légèrement défavorable, et c'est assumé.
- **Le jeu `test` de SciFact a ses étiquettes cachées.** Aucun chiffre sur `test`
  n'est possible sans passer par leur classement en ligne. Tous les repères cités
  sont les valeurs `dev` de la table 7.
- **`dev` compte 300 affirmations.** Un écart de un ou deux points n'est pas
  significatif. La cellule finale le teste au lieu de l'affirmer.
- **La récupération reste BM25 seul**, plafond 89,7 au top-3. L'hybride mesuré à
  0,8304 monterait ce plafond, mais il demande GTE et donc `transformers<5`,
  incompatible avec l'entraînement fait ici. C'est un choix, pas un oubli.
- **Aucune recherche d'hyperparamètres au-delà de ce qui est balayé ici.** Taux
  d'apprentissage et nombre d'époques sont ceux du papier.

## 1. Environnement

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "AUCUN GPU")

In [ ]:
!pip -q install transformers torch nltk pandas sentencepiece 2>&1 | tail -2
import torch, transformers, numpy as np, json, os, random, time
from collections import Counter
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())
GRAINE = 0
random.seed(GRAINE); np.random.seed(GRAINE); torch.manual_seed(GRAINE)
torch.cuda.manual_seed_all(GRAINE)

## 2. Données, évaluateur officiel, découpage train / val / dev

In [ ]:
import tarfile, zipfile, urllib.request

if not os.path.exists("data/claims_dev.jsonl"):
    urllib.request.urlretrieve(
        "https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz", "v.tar.gz")
    tarfile.open("v.tar.gz").extractall(".")

os.makedirs("evaluate/lib", exist_ok=True)
B = "https://raw.githubusercontent.com/allenai/scifact/master/verisci/evaluate"
urllib.request.urlretrieve(f"{B}/pipeline.py", "evaluate/pipeline.py")
for f in ["__init__.py", "data.py", "metrics.py"]:
    urllib.request.urlretrieve(f"{B}/lib/{f}", f"evaluate/lib/{f}")

corpus = {str(json.loads(l)["doc_id"]): json.loads(l)
          for l in open("data/corpus.jsonl", encoding="utf-8")}
train_all = [json.loads(l) for l in open("data/claims_train.jsonl", encoding="utf-8")]
dev = [json.loads(l) for l in open("data/claims_dev.jsonl", encoding="utf-8")]

random.Random(GRAINE).shuffle(train_all)
n_val = int(0.15 * len(train_all))
val, train = train_all[:n_val], train_all[n_val:]

# val doit servir a evaluer la chaine complete : il lui faut son propre fichier d'or
with open("data/claims_val.jsonl", "w") as f:
    for x in val: f.write(json.dumps(x) + "\n")

print(f"train {len(train)} | val {len(val)} (réglage) | dev {len(dev)} (rapport)")
assert len(corpus) == 5183 and len(dev) == 300

## 3. Récupération BM25 — inchangée, contrôlée à 0,6756 nDCG@10

In [ ]:
import re
from nltk.stem.porter import PorterStemmer

ARRET = set('''a an and are as at be but by for if in into is it no not of on or
such that the their then there these they this to was will with'''.split())
_r, _cache = PorterStemmer(), {}
def normaliser(t):
    out = []
    for m in re.findall(r"[a-z0-9]+", t.lower()):
        if m in ARRET: continue
        s = _cache.get(m)
        if s is None: s = _r.stem(m); _cache[m] = s
        out.append(s)
    return out

class BM25:
    def __init__(self, docs, k1=0.9, b=0.4):
        self.k1, self.b = k1, b
        j = [normaliser(d) for d in docs]
        self.n = len(j)
        self.lg = np.array([len(d) for d in j], dtype=np.float32)
        self.moy = float(self.lg.mean())
        brut = {}
        for i, doc in enumerate(j):
            for t, f in Counter(doc).items(): brut.setdefault(t, []).append((i, f))
        self.index = {}
        for t, post in brut.items():
            idx = np.array([p[0] for p in post], dtype=np.int32)
            frq = np.array([p[1] for p in post], dtype=np.float32)
            df = len(post)
            self.index[t] = (idx, frq, float(np.log(1 + (self.n - df + .5) / (df + .5))))
    def scores(self, q):
        s = np.zeros(self.n, dtype=np.float32)
        for t in normaliser(q):
            e = self.index.get(t)
            if e is None: continue
            idx, frq, idf = e
            norme = 1 - self.b + self.b * self.lg[idx] / self.moy
            s[idx] += idf * (frq * (self.k1 + 1)) / (frq + self.k1 * norme)
        return s

doc_ids = list(corpus)
lex = BM25([f"{corpus[d]['title']} {' '.join(corpus[d]['abstract'])}" for d in doc_ids])

def recuperer(claims, k):
    out = {}
    for c in claims:
        s = lex.scores(c["claim"])
        top = np.argpartition(-s, k)[:k]
        out[c["id"]] = [doc_ids[i] for i in top[np.argsort(-s[top])]]
    return out

K_MAX = 10                      # on recupere large, on tronquera au balayage
rec = {"val": recuperer(val, K_MAX), "dev": recuperer(dev, K_MAX)}

def couverture(claims, r, k):
    vp = att = 0
    for c in claims:
        if not c.get("evidence"): continue
        vp += len(set(c["evidence"]) & set(r[c["id"]][:k]))
        att += len(c["evidence"])
    return vp / att
for k in (3, 5, 10):
    print(f"couverture dev au top-{k:>2} : {couverture(dev, rec['dev'], k):.3f}")

## 4. Entraînement — utilitaires

`lr_scheduler.step()` est appelé **après** `optimizer.step()`, et le modèle est
forcé en fp32 : transformers récent respecte le `torch_dtype` du checkpoint, et
un modèle chargé en fp16 fait échouer `GradScaler.unscale_`.

In [ ]:
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

def charger(nom, n_labels=None):
    kw = {"num_labels": n_labels} if n_labels else {}
    m = AutoModelForSequenceClassification.from_pretrained(nom, **kw)
    return m.float().to("cuda")          # fp32 obligatoire pour GradScaler

class Paires(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], int(self.y[i])

def assembler(tok, lot, longueur):
    X = [b[0] for b in lot]; y = torch.tensor([b[1] for b in lot])
    e = tok([a for a, _ in X], [b for _, b in X], padding=True, truncation=True,
            max_length=longueur, return_tensors="pt")
    return e, y

def entrainer(modele, tok, X, y, epoques=3, lot=32, lr=2e-5, longueur=256, poids=None):
    dl = DataLoader(Paires(X, y), batch_size=lot, shuffle=True,
                    collate_fn=lambda b: assembler(tok, b, longueur))
    opt = torch.optim.AdamW(modele.parameters(), lr=lr, weight_decay=0.01)
    total = len(dl) * epoques
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    perte = torch.nn.CrossEntropyLoss(weight=poids.to("cuda") if poids is not None else None)
    scaler = torch.amp.GradScaler("cuda")
    modele.train()
    for ep in range(epoques):
        cumul, t0 = 0.0, time.time()
        for e, cible in dl:
            e = {k: v.to("cuda") for k, v in e.items()}; cible = cible.to("cuda")
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                p = perte(modele(**e).logits, cible)
            scaler.scale(p).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(modele.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            sched.step()                  # apres optimizer.step()
            cumul += p.item()
        print(f"  époque {ep+1}/{epoques} — perte {cumul/len(dl):.4f} ({time.time()-t0:.0f} s)")
    modele.eval()
    return modele

@torch.no_grad()
def noter(modele, tok, X, lot=128, longueur=256, colonne=1):
    out = []
    for i in range(0, len(X), lot):
        p = X[i:i + lot]
        e = tok([a for a, _ in p], [b for _, b in p], padding=True, truncation=True,
                max_length=longueur, return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            z = torch.softmax(modele(**e).logits.float(), -1)
        out.append(z[:, colonne].cpu().numpy() if colonne is not None else z.cpu().numpy())
    if not out:
        return np.zeros(0)
    return np.concatenate(out) if colonne is not None else np.vstack(out)

## 5. Sélecteur de justifications — deux candidats, le meilleur retenu sur `val`

SciBERT est pré-entraîné sur du texte scientifique et donne le meilleur F1 de
sélection dans le papier (74,4). DeBERTa-v3-base est plus récent mais générique
(65,0 à l'essai précédent). On tranche sur `val`, pas à l'intuition.

In [ ]:
def donnees_selection(claims):
    X, y = [], []
    for c in claims:
        ev = c.get("evidence") or {}
        for doc in c.get("cited_doc_ids", []):
            doc = str(doc)
            if doc not in corpus: continue
            ors = {s for g in ev.get(doc, []) for s in g["sentences"]}
            for i, ph in enumerate(corpus[doc]["abstract"]):
                X.append((ph, c["claim"])); y.append(1 if i in ors else 0)
    return X, np.array(y)

Xtr, ytr = donnees_selection(train)
Xva, yva = donnees_selection(val)
Xdev_or, ydev_or = donnees_selection(dev)
print(f"sélection — train {len(Xtr)} phrases, {100*ytr.mean():.1f} % justificatives")

def prf(scores, y, t):
    p = scores >= t
    vp = int((p & (y == 1)).sum()); fp = int((p & (y == 0)).sum()); fn = int((~p & (y == 1)).sum())
    P = vp / (vp + fp) if vp + fp else 0.0
    R = vp / (vp + fn) if vp + fn else 0.0
    return P, R, 2 * P * R / (P + R) if P + R else 0.0

CANDIDATS = ["allenai/scibert_scivocab_uncased", "microsoft/deberta-v3-base"]
poids = torch.tensor([1.0, float((ytr == 0).sum() / max((ytr == 1).sum(), 1))])
print(f"poids de la classe positive : {poids[1]:.2f}\n")

resultats_sel = {}
for nom in CANDIDATS:
    print(f"=== {nom} ===")
    tk = AutoTokenizer.from_pretrained(nom)
    md_ = charger(nom, n_labels=2)
    md_ = entrainer(md_, tk, Xtr, ytr, epoques=3, lot=32, lr=2e-5, poids=poids)
    sv = noter(md_, tk, Xva)
    t_best, f_best = max(((t, prf(sv, yva, t)[2]) for t in np.arange(0.05, 0.96, 0.05)),
                         key=lambda x: x[1])
    resultats_sel[nom] = {"modele": md_, "tok": tk, "seuil_val": float(t_best),
                          "f1_val": float(f_best)}
    print(f"  F1 sur val : {100*f_best:.1f} (seuil {t_best:.2f})\n")

MEILLEUR = max(resultats_sel, key=lambda n: resultats_sel[n]["f1_val"])
sel, tok_sel = resultats_sel[MEILLEUR]["modele"], resultats_sel[MEILLEUR]["tok"]
SEUIL_SEL_INIT = resultats_sel[MEILLEUR]["seuil_val"]
print(f"retenu : {MEILLEUR}")

# Liberer les perdants : sinon le classifieur large ne tient pas en memoire.
for nom in list(resultats_sel):
    if nom != MEILLEUR:
        del resultats_sel[nom]["modele"]
        del resultats_sel[nom]
import gc
gc.collect(); torch.cuda.empty_cache()
print(f"mémoire GPU occupée : {torch.cuda.memory_allocated()/1e9:.2f} Go")

P, R, F = prf(noter(sel, tok_sel, Xdev_or), ydev_or, SEUIL_SEL_INIT)
print(f"\nCONTRÔLE — sélection sur dev, abstracts d'or : P {100*P:.1f} R {100*R:.1f} F1 {100*F:.1f}")
print(f"  repère RoBERTa-large : F1 72,1  |  SciBERT : F1 74,4")

## 6. Classifieur d'étiquette — entraîné sur les sorties **réelles** du sélecteur

C'est la correction principale. Chaque exemple d'entraînement est construit comme
à l'inférence : le sélecteur choisit les phrases, et l'étiquette d'or de
l'abstract sert de cible. Plus de raccourci « trois premières phrases = NOINFO ».

In [ ]:
MAX_PHRASES = 3   # plafond du code d'evaluation officiel

def phrases_choisies(claim, doc, seuil):
    # Phrases au-dessus du seuil, sinon la mieux notee.
    # Le repli evite d ecarter l abstract : c est le classifieur, entraine pour
    # cela, qui decidera s il constitue une preuve ou non. Sans ce repli, les
    # abstracts NOINFO disparaissaient de l entrainement : 4 exemples sur 477.
    ph = corpus[doc]["abstract"]
    if not ph:
        return []
    z = noter(sel, tok_sel, [(p, claim) for p in ph])
    idx = np.where(z >= seuil)[0]
    if len(idx) == 0:
        idx = np.array([int(z.argmax())])
    idx = idx[np.argsort(-z[idx])][:MAX_PHRASES]
    return sorted(int(i) for i in idx)

def donnees_etiquette_reelles(claims, seuil):
    X, y, ignores = [], [], 0
    for c in claims:
        ev = c.get("evidence") or {}
        for doc in c.get("cited_doc_ids", []):
            doc = str(doc)
            if doc not in corpus: continue
            idx = phrases_choisies(c["claim"], doc, seuil)
            if not idx:
                ignores += 1      # abstract vide, cas pathologique
                continue
            groupes = ev.get(doc)
            X.append((" ".join(corpus[doc]["abstract"][i] for i in idx), c["claim"]))
            y.append(groupes[0]["label"] if groupes else "NOINFO")
    return X, y, ignores

MODELE_ETIQ = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
tok_etiq = AutoTokenizer.from_pretrained(MODELE_ETIQ)
etiq = charger(MODELE_ETIQ)

id2label = {int(k): v.lower() for k, v in etiq.config.id2label.items()}
I_POUR   = next(i for i, v in id2label.items() if v.startswith("entail"))
I_NEUTRE = next(i for i, v in id2label.items() if v.startswith("neutral"))
I_CONTRE = next(i for i, v in id2label.items() if v.startswith("contradic"))
VERS_INDICE = {"SUPPORT": I_POUR, "NOINFO": I_NEUTRE, "CONTRADICT": I_CONTRE}
VERS_NOM = {v: k for k, v in VERS_INDICE.items()}
print("association :", {k: id2label[v] for k, v in VERS_INDICE.items()})

Xe_tr, noms_tr, ign = donnees_etiquette_reelles(train, SEUIL_SEL_INIT)
ye_tr = np.array([VERS_INDICE[n] for n in noms_tr])
print(f"\nétiquette — {len(Xe_tr)} exemples construits par le sélecteur "
      f"({ign} abstracts écartés)")
assert noms_tr.count("NOINFO") >= 50, (
    f"seulement {noms_tr.count('NOINFO')} exemples NOINFO : le classifieur ne "
    "pourra pas apprendre a filtrer. Verifier le repli sur la meilleure phrase.")
print("  répartition :", {n: noms_tr.count(n) for n in set(noms_tr)})

etiq = entrainer(etiq, tok_etiq, Xe_tr, ye_tr, epoques=3, lot=4, lr=1e-5, longueur=320)

@torch.no_grad()
def classer_etiq(X, lot=16):
    return noter(etiq, tok_etiq, X, lot=lot, longueur=320, colonne=None)

# Controle honnete : memes conditions qu'a l'inference, sur dev.
Xe_dev, noms_dev, _ = donnees_etiquette_reelles(dev, SEUIL_SEL_INIT)
if Xe_dev:
    pred = classer_etiq(Xe_dev).argmax(1)
    cible = np.array([VERS_INDICE[n] for n in noms_dev])
    print(f"\nCONTRÔLE — étiquette sur dev, entrées du sélecteur : "
          f"exactitude {100*(pred == cible).mean():.1f}")
    print("  repère RoBERTa-large : 75,7  |  SciBERT : 69,2")
    print("  (le papier mesure avec les justifications d'or ; ici c'est plus dur,")
    print("   mais c'est la condition reelle de la chaine)")
    for nom, i in VERS_INDICE.items():
        m = cible == i
        if m.sum():
            print(f"  {nom:<11}{int(m.sum()):>5} exemples, exactitude {100*(pred[m] == i).mean():.1f}")

## 7. Chaîne complète, et réglage sur `val` de ce qui compte vraiment

On balaie le seuil de sélection **et** le nombre d'abstracts, en optimisant le
F1 au niveau abstract — la métrique du rapport — et non le F1 de sélection de
phrases comme la fois précédente.

In [ ]:
def chaine(claims, r, seuil, k):
    sortie = []
    for c in claims:
        preuve = {}
        for doc in r[c["id"]][:k]:
            idx = phrases_choisies(c["claim"], doc, seuil)
            if not idx: continue
            texte = " ".join(corpus[doc]["abstract"][i] for i in idx)
            nom = VERS_NOM[int(classer_etiq([(texte, c["claim"])])[0].argmax())]
            if nom != "NOINFO":
                preuve[str(doc)] = {"label": nom, "sentences": idx}
        sortie.append({"id": c["id"], "evidence": preuve})
    return sortie

def evaluer_officiel(predictions, gold):
    with open("pred.jsonl", "w") as f:
        for p in predictions: f.write(json.dumps(p) + "\n")
    r = subprocess.run(["python", "pipeline.py", "--gold", f"../{gold}",
                        "--corpus", "../data/corpus.jsonl", "--prediction", "../pred.jsonl",
                        "--output", "../m.json"], cwd="evaluate", capture_output=True, text=True)
    if not os.path.exists("m.json"):
        print(r.stdout[-1200:], r.stderr[-1200:]); raise RuntimeError("évaluateur en échec")
    m = json.load(open("m.json")); os.remove("m.json")
    return m

print(f"{'seuil':>7}{'k':>4}{'abstract F1 (val)':>20}")
print("-" * 31)
meilleur = (None, -1.0)
for seuil in (0.30, 0.50, 0.70, 0.85, 0.95):
    for k in (3, 5):
        m = evaluer_officiel(chaine(val, rec["val"], seuil, k), "data/claims_val.jsonl")
        f = m["abstract_rationalized"]["f1"] * 100
        if f > meilleur[1]:
            meilleur = ((seuil, k), f)
        print(f"{seuil:>7.2f}{k:>4}{f:>20.1f}")

(SEUIL, K_ABS), _ = meilleur
print(f"\nretenu sur val : seuil {SEUIL}, k {K_ABS}")

## 8. Résultat sur `dev` — une seule mesure, avec les réglages figés

In [ ]:
predictions_dev = chaine(dev, rec["dev"], SEUIL, K_ABS)
m = evaluer_officiel(predictions_dev, "data/claims_dev.jsonl")
ph = m["sentence_label"]["f1"] * 100
ab = m["abstract_rationalized"]["f1"] * 100

REPERES = [
    ("Zéro-shot (FEVER), 2020",     28.4, 38.4),
    ("ce système, zéro-shot",       26.6, 35.3),
    ("ce système, affinage v1",     39.2, 48.0),
    ("VeriSci, régime ouvert",      42.6, 48.5),
    ("VeriSci, abstracts fournis",  60.6, 72.5),
    ("Justifications fournies",     79.9, 83.0),
]
print(f"{'système':<36}{'phrase':>10}{'abstract':>11}")
print("-" * 57)
for nom, p, a in REPERES:
    print(f"{nom:<36}{p:>10.1f}{a:>11.1f}")
print("-" * 57)
marque = "  ← dépasse VeriSci" if ab > 48.5 else ""
print(f"{'ce système, affinage v2':<36}{ph:>10.1f}{ab:>11.1f}{marque}")
print("-" * 57)
print(f"{'plafond de la récupération':<36}{'':>10}{89.7:>11.1f}")
print()
for cle, lib in [("sentence_selection", "phrase, sélection seule"),
                 ("sentence_label", "phrase, sélection + étiquette"),
                 ("abstract_label_only", "abstract, étiquette seule"),
                 ("abstract_rationalized", "abstract, étiquette + justification")]:
    d = m[cle]
    print(f"  {lib:<38} P {d['precision']*100:>5.1f}  R {d['recall']*100:>5.1f}  F1 {d['f1']*100:>5.1f}")

## 9. L'écart avec VeriSci est-il réel ?

300 affirmations. Un ou deux points ne veulent rien dire sans test. Bootstrap
apparié sur les affirmations, contre la valeur publiée de VeriSci.

In [ ]:
# Le F1 au niveau abstract est global, pas moyennable par affirmation : on
# reechantillonne les affirmations et on recalcule le F1 sur chaque tirage.
def f1_abstract(preds, claims):
    vp = fp = fn = 0
    par_id = {p["id"]: p for p in preds}
    for c in claims:
        gold = c.get("evidence") or {}
        pred = par_id[c["id"]]["evidence"]
        for doc, d in pred.items():
            g = gold.get(doc)
            ok = False
            if g and g[0]["label"] == d["label"]:
                ok = any(set(gr["sentences"]) <= set(d["sentences"]) for gr in g)
            vp += ok
            fp += not ok
        for doc in gold:
            if doc not in pred:
                fn += 1
    P = vp / (vp + fp) if vp + fp else 0.0
    R = vp / (vp + fn) if vp + fn else 0.0
    return 2 * P * R / (P + R) if P + R else 0.0

base = f1_abstract(predictions_dev, dev) * 100
print(f"F1 recalculé localement : {base:.1f}  (officiel : {ab:.1f})")

rng = np.random.default_rng(0)
tirages = []
for _ in range(2000):
    ech = rng.choice(len(dev), len(dev), replace=True)
    tirages.append(f1_abstract(predictions_dev, [dev[i] for i in ech]) * 100)
tirages = np.array(tirages)
ic = np.percentile(tirages, [2.5, 97.5])
print(f"IC 95 % : [{ic[0]:.1f}, {ic[1]:.1f}]")
dedans = ic[0] <= 48.5 <= ic[1]
print(f"VeriSci publié : 48,5 — {'dans' if dedans else 'hors de'} l'intervalle")
print("→", "écart non significatif" if dedans else "écart significatif")

json.dump({"selecteur": MEILLEUR, "classifieur": MODELE_ETIQ,
           "seuil": SEUIL, "k_abstracts": K_ABS,
           "metriques_dev": m, "ic95_abstract_f1": list(ic),
           "reperes": {n: {"phrase": p, "abstract": a} for n, p, a in REPERES}},
          open("resultats_affinage_v2.json", "w"), indent=2)
print("\nécrit : resultats_affinage_v2.json")

## 10. Récupérer

In [ ]:
from google.colab import files
files.download("resultats_affinage_v2.json")